# Ceeria Workflow — Tool Calling (Function Calling) 버전

## 키워드 방식 vs Tool Calling 방식

| | 키워드 방식 (기존) | Tool Calling 방식 (이 노트북) |
|---|---|---|
| 선택 주체 | `if/elif` 조건문 | LLM |
| 확장성 | 키워드 추가 필요 | 도구 설명만 추가 |
| 모호한 질문 | 매칭 실패 | LLM이 의도 추론 |
| 비용 | 없음 | LLM 호출 비용 발생 |

## Tool Calling 동작 원리

```
사용자 질문
    │
    ▼
[LLM + bind_tools]  ← 도구 목록(이름+설명+파라미터) 전달
    │
    ▼  LLM이 어떤 도구를 어떤 인자로 호출할지 결정
[tool_calls 파싱]
    │
    ▼
[ToolNode 실행]  ← 선택된 함수 실제 실행
    │
    ▼
[generate_response]  ← 결과를 바탕으로 최종 답변 생성
```

---
**이 노트북은 외부 API를 Mock 데이터로 대체하여 로컬에서 실행 가능합니다.**  
실제 LLM 호출 부분은 `OPENAI_API_KEY` 또는 사내 LLM 설정이 필요합니다.

## 1. 라이브러리 임포트

In [1]:
import re
import json
import os
import functools
from typing import List, Optional, Literal, Dict, Any, Annotated
from collections import defaultdict
from pydantic import BaseModel, Field
from IPython.display import Markdown, display

# LangChain / LangGraph
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI

print("임포트 완료")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


임포트 완료


## 2. 샘플 데이터 & Mock API (기존과 동일)

In [2]:
# ── EQP 샘플 ──────────────────────────────────────────────────
SAMPLE_EQP_M15 = [
    {"EQP_ID": "M15A001", "FAC_ID": "M15", "DET_FAC_ID": "M15A",
     "EQP_MODEL_CD": "LPCVD_A", "EQ_GROUP": "CVD", "VENDOR_NM": "AMAT",
     "MGMT_AREA_ID": "CVD", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Up", "EQP_STAT_CD": "RUN",
     "BAY_NM": "BAY-01", "LAST_EVENT_TM": "2026-05-07 08:00:00",
     "PORT_INFO": "P1l진행중lRun,P2l대기lIdle"},
    {"EQP_ID": "M15A002", "FAC_ID": "M15", "DET_FAC_ID": "M15A",
     "EQP_MODEL_CD": "LPCVD_A", "EQ_GROUP": "CVD", "VENDOR_NM": "AMAT",
     "MGMT_AREA_ID": "CVD", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Down", "EQP_STAT_CD": "PM",
     "BAY_NM": "BAY-01", "LAST_EVENT_TM": "2026-05-06 14:30:00", "PORT_INFO": ""},
    {"EQP_ID": "M15B001", "FAC_ID": "M15", "DET_FAC_ID": "M15B",
     "EQP_MODEL_CD": "CMP_B", "EQ_GROUP": "CMP", "VENDOR_NM": "KLA",
     "MGMT_AREA_ID": "CMP", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Up", "EQP_STAT_CD": "RUN",
     "BAY_NM": "BAY-02", "LAST_EVENT_TM": "2026-05-07 09:15:00", "PORT_INFO": ""},
    {"EQP_ID": "M15A001_CH1", "FAC_ID": "M15", "DET_FAC_ID": "M15A",
     "EQP_MODEL_CD": "LPCVD_A", "EQ_GROUP": "CVD", "VENDOR_NM": "AMAT",
     "MGMT_AREA_ID": "CVD", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Up", "EQP_STAT_CD": "RUN",
     "BAY_NM": "BAY-01", "LAST_EVENT_TM": "2026-05-07 08:00:00", "PORT_INFO": ""},
    {"EQP_ID": "M15A001_CH2", "FAC_ID": "M15", "DET_FAC_ID": "M15A",
     "EQP_MODEL_CD": "LPCVD_A", "EQ_GROUP": "CVD", "VENDOR_NM": "AMAT",
     "MGMT_AREA_ID": "CVD", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Down", "EQP_STAT_CD": "FAULT",
     "BAY_NM": "BAY-01", "LAST_EVENT_TM": "2026-05-06 20:00:00", "PORT_INFO": ""},
]
SAMPLE_EQP_M16 = [
    {"EQP_ID": "M16C001", "FAC_ID": "M16", "DET_FAC_ID": "M16C",
     "EQP_MODEL_CD": "ALD_C", "EQ_GROUP": "ALD", "VENDOR_NM": "TEL",
     "MGMT_AREA_ID": "ALD", "SECTION_GRP_NM": "M16DRAM",
     "MES_STAT_TYP": "Up", "EQP_STAT_CD": "RUN",
     "BAY_NM": "BAY-03", "LAST_EVENT_TM": "2026-05-07 07:00:00", "PORT_INFO": ""},
    {"EQP_ID": "M16C002", "FAC_ID": "M16", "DET_FAC_ID": "M16C",
     "EQP_MODEL_CD": "ALD_C", "EQ_GROUP": "ALD", "VENDOR_NM": "TEL",
     "MGMT_AREA_ID": "ALD", "SECTION_GRP_NM": "M16DRAM",
     "MES_STAT_TYP": "Down", "EQP_STAT_CD": "PM",
     "BAY_NM": "BAY-03", "LAST_EVENT_TM": "2026-05-06 18:00:00", "PORT_INFO": ""},
]
SAMPLE_LOT_DATA = {
    "AB123456": [{"LOT_ID": "AB123456", "PROD_ID": "NAND_128G", "STATUS": "ACTIVE"}],
    "CD789012": [{"LOT_ID": "CD789012", "PROD_ID": "DRAM_16G",  "STATUS": "ACTIVE"}],
}
SAMPLE_SETMO = [
    {"LOT_ID": "AB123456", "ACT_NM": "SetMonitor",
     "OPER_DESC": "CVD 증착", "ACT_DESC": "두께 측정 필요",
     "MEAS_SLOT_NM": "Slot 1,3,5", "ENGR_USER_NM": "김엔지니어"},
    {"LOT_ID": "AB123456", "ACT_NM": "SetMonitor",
     "OPER_DESC": "CMP 연마", "ACT_DESC": "평탄도 확인",
     "MEAS_SLOT_NM": "Slot 2,4", "ENGR_USER_NM": "이엔지니어"},
    {"LOT_ID": "AB123456", "ACT_NM": "makeOnHold",
     "OPER_DESC": "식각 공정", "ACT_DESC": "이상 감지 시 홀드",
     "MEAS_SLOT_NM": "-", "ENGR_USER_NM": "박엔지니어"},
]
SAMPLE_SLOT = [
    {"POSITION_VAL": 1,  "WF_ID": "WF001", "FAB": "M15", "OPER_DESC": "CVD 증착",  "EVENT_TM": "2026-05-07 08:00:00"},
    {"POSITION_VAL": 2,  "WF_ID": "WF002", "FAB": "M15", "OPER_DESC": "CVD 증착",  "EVENT_TM": "2026-05-07 08:01:00"},
    {"POSITION_VAL": 3,  "WF_ID": "WF003", "FAB": "M15", "OPER_DESC": "CMP 연마",  "EVENT_TM": "2026-05-07 08:02:00"},
    {"POSITION_VAL": 5,  "WF_ID": "WF005", "FAB": "M15", "OPER_DESC": "검사",      "EVENT_TM": "2026-05-07 08:03:00"},
]
SAMPLE_LOTHIS = [
    {"TIMEKEY": "20260507090000", "EVENT_CD": "TRACK_IN",  "OPER_ID": "CVD001",   "CTN_DESC": "CVD 증착",         "WF_QTY": 25, "PROD_ID": "NAND_128G"},
    {"TIMEKEY": "20260507080000", "EVENT_CD": "TRACK_OUT", "OPER_ID": "PHOTO001", "CTN_DESC": "포토 리소그래피",  "WF_QTY": 25, "PROD_ID": "NAND_128G"},
    {"TIMEKEY": "20260507060000", "EVENT_CD": "TRACK_IN",  "OPER_ID": "PHOTO001", "CTN_DESC": "포토 리소그래피",  "WF_QTY": 25, "PROD_ID": "NAND_128G"},
    {"TIMEKEY": "20260506200000", "EVENT_CD": "TRACK_OUT", "OPER_ID": "CMP001",   "CTN_DESC": "CMP 연마",        "WF_QTY": 25, "PROD_ID": "NAND_128G"},
]
SAMPLE_OPERHIS = [
    {"LOT_ID": "AB1001", "OPERATIONDESC": "CVD 증착",        "OPERLEVEL": 300, "WF_QTY": 25, "CTN_DESC": "CVD 증착",        "MES_PROC_STAT_CD": "COMPLETE", "LAST_EVENT_TM": "2026-05-07 08:00", "FLOW_ID": "FLOW_A"},
    {"LOT_ID": "AB1002", "OPERATIONDESC": "CVD 증착",        "OPERLEVEL": 290, "WF_QTY": 24, "CTN_DESC": "CVD 증착",        "MES_PROC_STAT_CD": "COMPLETE", "LAST_EVENT_TM": "2026-05-06 20:00", "FLOW_ID": "FLOW_A"},
    {"LOT_ID": "AB1001", "OPERATIONDESC": "포토 리소그래피",  "OPERLEVEL": 280, "WF_QTY": 25, "CTN_DESC": "포토 리소그래피", "MES_PROC_STAT_CD": "COMPLETE", "LAST_EVENT_TM": "2026-05-07 06:00", "FLOW_ID": "FLOW_A"},
    {"LOT_ID": "AB1001", "OPERATIONDESC": "CMP 연마",        "OPERLEVEL": 270, "WF_QTY": 25, "CTN_DESC": "CMP 연마",        "MES_PROC_STAT_CD": "COMPLETE", "LAST_EVENT_TM": "2026-05-06 18:00", "FLOW_ID": "FLOW_A"},
]

# ── Mock API ──────────────────────────────────────────────────
def call_lotid(lot_id):        return SAMPLE_LOT_DATA.get(lot_id.upper(), [])
def call_setmo_check(lot_id):  return [d for d in SAMPLE_SETMO if d["LOT_ID"] == lot_id.upper()]
def call_slot_info(lot_id):    return SAMPLE_SLOT
def call_lothis(lot_id):       return SAMPLE_LOTHIS
def call_eq(eqp_id):           return [d for d in SAMPLE_EQP_M15 + SAMPLE_EQP_M16 if d["EQP_ID"] == eqp_id.upper()]
def call_eqpm15(eqp_id="*"): return SAMPLE_EQP_M15 if eqp_id == "*" else [d for d in SAMPLE_EQP_M15 if d["EQP_ID"] == eqp_id]
def call_eqpm16(eqp_id="*"): return SAMPLE_EQP_M16 if eqp_id == "*" else [d for d in SAMPLE_EQP_M16 if d["EQP_ID"] == eqp_id]
def call_eqpm10(eqp_id="*"): return []
def call_eqpm11(eqp_id="*"): return []
def call_eqpm14(eqp_id="*"): return []
def call_operhis_api(lot_cd):  return SAMPLE_OPERHIS

print("샘플 데이터 & Mock API 준비 완료")

샘플 데이터 & Mock API 준비 완료


## 3. GraphState & 공통 헬퍼 함수

In [3]:
class GraphState(BaseModel):
    query: str = ""
    answer: Optional[str] = None
    intent: Optional[str] = None
    is_eqp: bool = False
    is_lot: bool = False
    eqp_id: Optional[str] = None
    lot_id: Optional[str] = None
    fab: Optional[str] = None
    skip_rag: bool = False
    tool_calls_log: List[str] = Field(default_factory=list)
    class Config:
        arbitrary_types_allowed = True


@functools.lru_cache(maxsize=32)
def get_eqp_by_fab(fab_input: str, eqp_id: str = "*") -> tuple:
    fab_key = fab_input.upper().replace('X', '')
    api_map = {'M10': call_eqpm10, 'M11': call_eqpm11, 'M14': call_eqpm14,
               'M15': call_eqpm15, 'M16': call_eqpm16}
    result = api_map.get(fab_key, lambda x: [])(eqp_id) or []
    return tuple(json.dumps(d) for d in result)  # lru_cache는 hashable 필요

def get_eqp_list(fab_input: str, eqp_id: str = "*") -> List[Dict]:
    return [json.loads(s) for s in get_eqp_by_fab(fab_input, eqp_id)]


def format_setmo_data(data, lot_id, filter_act=None):
    if not data: return f"{lot_id} SETMO 정보 없음"
    if filter_act: data = [d for d in data if d.get('ACT_NM') == filter_act]
    if not data: return f"{lot_id} 해당 SETMO 정보 없음"
    if filter_act == "makeOnHold":
        lines = [f"##{lot_id} - Future Hold\n", "| 공정 | Comment | 엔지니어 |", "| --- | --- | --- |"]
        for d in data[:20]: lines.append(f"| {d.get('OPER_DESC','-')} | {d.get('ACT_DESC','-')} | {d.get('ENGR_USER_NM','-')} |")
    else:
        lines = [f"##{lot_id} - SetMonitor\n", "| 공정 | Comment | 측정 Slot | 엔지니어 |", "| --- | --- | --- | --- |"]
        for d in data[:20]: lines.append(f"| {d.get('OPER_DESC','-')} | {d.get('ACT_DESC','-')} | {d.get('MEAS_SLOT_NM','-')} | {d.get('ENGR_USER_NM','-')} |")
    return '\n'.join(lines)

def format_slot_data(data, lot_id):
    if not data: return f"{lot_id} 슬롯 정보 없음"
    latest = {}
    for d in data:
        p = d.get("POSITION_VAL")
        if p is not None:
            p = int(p)
            if p not in latest or d.get('EVENT_TM','') > latest[p].get('EVENT_TM',''):
                latest[p] = d
    lines = [f"##{lot_id} - Slot 정보\n", "| Slot | WF_ID | 공정 (FAB) | 시간 |", "| --- | --- | --- | --- |"]
    for i in range(1, 26):
        if i in latest:
            d = latest[i]
            lines.append(f"| {i} | {d.get('WF_ID','-')} | {d.get('OPER_DESC','-')} ({d.get('FAB','-')}) | {str(d.get('EVENT_TM','-'))[:19]} |")
        else:
            lines.append(f"| {i} | *Empty* | - | - |")
    return '\n'.join(lines)

def format_lothis_data(data, lot_id):
    if not data: return f"{lot_id} 이력 없음"
    sorted_data = sorted(data, key=lambda x: x.get('TIMEKEY',''), reverse=True)[:10]
    lines = [f"##{lot_id} - 이력정보\n", "| 순번 | 시간 | 이벤트 | 공정 ID | 공정명 | 수량 | 제품 |", "| --- | --- | --- | --- | --- | --- | --- |"]
    for i, d in enumerate(sorted_data, 1):
        t = str(d.get('TIMEKEY','-'))
        if len(t) >= 14 and t.isdigit():
            t = f"{t[:4]}-{t[4:6]}-{t[6:8]} {t[8:10]}:{t[10:12]}:{t[12:14]}"
        lines.append(f"| {i} | {t} | {d.get('EVENT_CD','-')} | {d.get('OPER_ID','-')} | {d.get('CTN_DESC','-')} | {d.get('WF_QTY','-')} | {d.get('PROD_ID','-')} |")
    return '\n'.join(lines)

def analyze_eqp_data(data, eqp_id):
    if not data: return {}
    main_eqp = next((d for d in data if d.get('EQP_ID') == eqp_id), data[0])
    chambers = [d for d in data if '_' in d.get('EQP_ID','') or d.get('EQP_ID','').startswith(eqp_id) and d.get('EQP_ID') != eqp_id]
    down_ch = [c for c in chambers if c.get('MES_STAT_TYP') == 'Down']
    port_list = []
    for seg in (main_eqp.get('PORT_INFO','') or '').split(','):
        parts = seg.strip().split('l')
        if len(parts) >= 2:
            port_list.append({'port': parts[0], 'transfer': parts[1] if len(parts)>1 else '-', 'status': parts[2] if len(parts)>2 else '-'})
    return {'EQP_ID': main_eqp.get('EQP_ID', eqp_id), 'EQ_GROUP': main_eqp.get('EQ_GROUP','-'),
            'MES_STAT_TYP': main_eqp.get('MES_STAT_TYP','-'), 'EQP_STAT_CD': main_eqp.get('EQP_STAT_CD','-'),
            'FAB': main_eqp.get('FAC_ID','-'), '장비사': main_eqp.get('VENDOR_NM','-'), '모델': main_eqp.get('EQP_MODEL_CD','-'),
            'CHAMBER 수': len(chambers), 'DOWN 수': len(down_ch), 'chamber_details': chambers, 'port_list': port_list}

def format_eqp_table(analyzed, show_chamber=True, show_port=True):
    lines = ["## 장비 상태\n", "| 항목 | 값 |", "| --- | --- |"]
    for k in ['EQP_ID','EQ_GROUP','MES_STAT_TYP','EQP_STAT_CD','CHAMBER 수','DOWN 수']:
        lines.append(f"| {k} | {analyzed.get(k,'-')} |")
    if show_chamber and analyzed.get('chamber_details'):
        lines += ["\n| 챔버 ID | 상태 | EQP_STAT | LAST_EVENT |", "| --- | --- | --- | --- |"]
        for ch in sorted(analyzed['chamber_details'], key=lambda x: x.get('EQP_ID','')):
            lines.append(f"| {ch.get('EQP_ID','-')} | {ch.get('MES_STAT_TYP','-')} | {ch.get('EQP_STAT_CD','-')} | {ch.get('LAST_EVENT_TM','-')} |")
    if show_port and analyzed.get('port_list'):
        lines += ["\n### PORT 정보", "| PORT | Transfer | 상태 |", "| --- | --- | --- |"]
        for p in analyzed['port_list']:
            lines.append(f"| {p['port']} | {p['transfer']} | {p['status']} |")
    return '\n'.join(lines)

def format_fab_summary(data, fab, query):
    main = [e for e in data if '_' not in e.get('EQP_ID','') and not any(s in e.get('EQP_ID','') for s in ['CH','SPIN'])]
    ch   = [e for e in data if e not in main]
    models = defaultdict(lambda: {'TOTAL':0,'UP':0,'DOWN':0})
    for e in main:
        m = e.get('EQP_MODEL_CD','Unknown')
        models[m]['TOTAL'] += 1
        models[m]['UP' if e.get('MES_STAT_TYP')=='Up' else 'DOWN'] += 1
    lines = [f"## {fab} 장비 현황 ({query})\n",
             f"- 메인 장비 총: {len(main)}대 / 챔버: {len(ch)}개", "",
             "| MODEL | TOTAL | UP | DOWN |", "| --- | --- | --- | --- |"]
    for k, v in sorted(models.items()):
        lines.append(f"| {k} | {v['TOTAL']} | {v['UP']} | {v['DOWN']} |")
    return '\n'.join(lines)

def format_operhis_data(data, ctn_desc):
    groups = defaultdict(list)
    for d in data:
        if d.get('LOT_ID'): groups[d['LOT_ID']].append(d)
    selected = sorted([max(v, key=lambda x: x.get('OPERLEVEL',0)) for v in groups.values()],
                      key=lambda x: x.get('OPERLEVEL',0), reverse=True)[:10]
    lines = [f"## {ctn_desc} 공정 이력\n", "| LOT_ID | WF_QTY | CTN_DESC | 상태 | 시간 | FLOW_ID |", "| --- | --- | --- | --- | --- | --- |"]
    for d in selected:
        lines.append(f"| {d.get('LOT_ID','-')} | {d.get('WF_QTY','-')} | {d.get('CTN_DESC','-')} | {d.get('MES_PROC_STAT_CD','-')} | {d.get('LAST_EVENT_TM','-')} | {d.get('FLOW_ID','-')} |")
    return '\n'.join(lines)

print("GraphState & 헬퍼 함수 준비 완료")

GraphState & 헬퍼 함수 준비 완료


/var/folders/3_/_4rs10ps6577z2vsry31x7g00000gn/T/ipykernel_43188/1120928006.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class GraphState(BaseModel):


## 4. `@tool` 로 도구 정의

LLM에게 전달되는 **도구 목록**입니다. 함수명과 `docstring`이 LLM의 선택 근거가 됩니다.

In [4]:
@tool
def get_lot_slot_info(lot_id: str) -> str:
    """LOT의 현재 슬롯별 위치와 공정 정보를 조회합니다.
    사용자가 LOT이 어디 있는지, 슬롯 정보, 현재 공정 위치를 물어볼 때 사용합니다.
    예: 'AB123456 슬롯 정보', 'AB123456 어디 있어', 'AB123456 현재 공정'
    """
    data = call_slot_info(lot_id)
    return format_slot_data(data, lot_id)


@tool
def get_lot_setmonitor(lot_id: str) -> str:
    """LOT의 Set Monitoring(셋모) 계측 지시 정보를 조회합니다.
    사용자가 셋모, setmo, 계측, S/M, set monitoring을 언급할 때 사용합니다.
    예: 'AB123456 셋모 확인', 'AB123456 setmonitor 있어?'
    """
    data = call_setmo_check(lot_id)
    return format_setmo_data(data, lot_id, "SetMonitor")


@tool
def get_lot_future_hold(lot_id: str) -> str:
    """LOT의 Future Hold(퓨처홀드) 정보를 조회합니다.
    사용자가 F/H, future hold, 퓨처홀드, makeOnHold를 언급할 때 사용합니다.
    예: 'AB123456 future hold 있어?', 'AB123456 F/H 확인'
    """
    data = call_setmo_check(lot_id)
    return format_setmo_data(data, lot_id, "makeOnHold")


@tool
def get_lot_history(lot_id: str) -> str:
    """LOT의 공정 이력(이동 이력)을 시간 순으로 조회합니다.
    사용자가 LOT 이력, 처리 이력, 어떤 공정을 거쳤는지 물어볼 때 사용합니다.
    예: 'AB123456 이력', 'AB123456 처리 이력 알려줘'
    """
    data = call_lothis(lot_id)
    return format_lothis_data(data, lot_id)


@tool
def get_equipment_status(eqp_id: str, fab: str) -> str:
    """특정 장비 ID의 현재 상태, 챔버 정보, PORT 정보를 조회합니다.
    사용자가 특정 장비 ID를 언급하며 상태, 다운여부, 정보를 물어볼 때 사용합니다.
    예: 'M15A001 상태', 'M15A001 장비 정보', 'M15A002 다운이야?'
    """
    all_data = get_eqp_list(fab, "*")
    eqp_group = [d for d in all_data if d['EQP_ID'].startswith(eqp_id)]
    if not eqp_group:
        return f"{eqp_id} 장비를 찾을 수 없습니다."
    analyzed = analyze_eqp_data(eqp_group, eqp_id)
    is_cmp = any('CMP' in str(d.get('MGMT_AREA_ID','')).upper() for d in eqp_group)
    return format_eqp_table(analyzed, show_chamber=not is_cmp, show_port=True)


@tool
def get_fab_equipment_summary(fab: str, model: Optional[str] = None, area: Optional[str] = None) -> str:
    """FAB(공장) 전체 또는 특정 모델/구역의 장비 현황(대수, UP/DOWN)을 집계합니다.
    사용자가 FAB 번호(M10~M16)를 언급하며 장비 현황, 댓수, DOWN 현황을 물어볼 때 사용합니다.
    예: 'M15 장비 현황', 'M16 ALD 다운 몇 대야?', 'M15 LPCVD_A 상태'
    """
    data = get_eqp_list(fab, "*")
    if model:
        data = [d for d in data if d.get('EQP_MODEL_CD') == model]
    if area:
        data = [d for d in data if d.get('MGMT_AREA_ID') == area or d.get('EQ_GROUP') == area]
    if not data:
        return f"{fab} 조건에 맞는 장비 없음"
    return format_fab_summary(data, fab, f"{model or ''} {area or ''}")


@tool
def get_operation_history(lot_prefix: str, operation_name: str, is_previous: bool = False) -> str:
    """특정 공정(operation)을 거친 LOT들의 이력을 조회합니다.
    사용자가 특정 공정명을 언급하거나 이전 공정을 물어볼 때 사용합니다.
    예: 'AB1 CVD 증착 이력', 'AB1 포토 공정 이전 단계', 'CVD 증착 진행 LOT 현황'
    """
    raw = call_operhis_api(lot_prefix) or []
    target = operation_name.strip().upper()
    matched = [d for d in raw if str(d.get('OPERATIONDESC','')).strip().upper() == target]
    if not matched:
        return f"'{operation_name}' 공정 데이터 없음"
    if is_previous:
        max_level = max(d.get('OPERLEVEL',0) for d in matched)
        raw = [d for d in raw if d.get('OPERLEVEL',0) < max_level and str(d.get('LOT_ID','')).startswith(lot_prefix)]
        raw = sorted(raw, key=lambda x: x.get('OPERLEVEL',0), reverse=True)[:10]
        return format_operhis_data(raw, f"{operation_name} 이전 공정")
    return format_operhis_data(matched, operation_name)


@tool
def answer_general_question(query: str) -> str:
    """위의 어떤 도구도 해당하지 않는 일반적인 질문에 사용합니다.
    공정 지식, 장비 원리, 트러블슈팅 방법 등 RAG 검색이 필요한 질문에 사용합니다.
    예: 'CVD 공정에서 두께 편차 원인은?', 'CMP 스크래치 발생 원인'
    """
    return f"[RAG 검색 필요] '{query}' → 지식 베이스 검색 후 LLM 답변 생성 단계로 진행"


# 도구 목록
TOOLS = [
    get_lot_slot_info,
    get_lot_setmonitor,
    get_lot_future_hold,
    get_lot_history,
    get_equipment_status,
    get_fab_equipment_summary,
    get_operation_history,
    answer_general_question,
]

print(f"정의된 도구 수: {len(TOOLS)}개")
for t in TOOLS:
    desc_first_line = t.description.strip().split('\n')[0]
    print(f"  - {t.name:<30} {desc_first_line}")

정의된 도구 수: 8개
  - get_lot_slot_info              LOT의 현재 슬롯별 위치와 공정 정보를 조회합니다.
  - get_lot_setmonitor             LOT의 Set Monitoring(셋모) 계측 지시 정보를 조회합니다.
  - get_lot_future_hold            LOT의 Future Hold(퓨처홀드) 정보를 조회합니다.
  - get_lot_history                LOT의 공정 이력(이동 이력)을 시간 순으로 조회합니다.
  - get_equipment_status           특정 장비 ID의 현재 상태, 챔버 정보, PORT 정보를 조회합니다.
  - get_fab_equipment_summary      FAB(공장) 전체 또는 특정 모델/구역의 장비 현황(대수, UP/DOWN)을 집계합니다.
  - get_operation_history          특정 공정(operation)을 거친 LOT들의 이력을 조회합니다.
  - answer_general_question        위의 어떤 도구도 해당하지 않는 일반적인 질문에 사용합니다.


## 5. LLM 초기화 & 도구 바인딩

`bind_tools()`로 LLM에게 도구 스키마(이름+설명+파라미터)를 전달합니다.

In [6]:
# ── LLM 설정 ──────────────────────────────────────────────────
# 방법 1: OpenAI API (공개)
# llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])

# 방법 2: 사내 LLM (Private Endpoint)
# llm = ChatOpenAI(
#     model="your-model-name",
#     base_url="https://your-internal-endpoint",
#     api_key="your-key",
# )

# 방법 3: 환경변수 자동 인식

import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# 도구를 LLM에 바인딩
llm_with_tools = llm.bind_tools(TOOLS)

print("LLM + bind_tools 완료")
print(f"모델: {llm.model_name}")
print(f"바인딩된 도구 수: {len(TOOLS)}개")

LLM + bind_tools 완료
모델: gpt-4o-mini
바인딩된 도구 수: 8개


## 6. Tool Calling 실행기

LLM의 `tool_calls` 응답을 파싱하고 해당 함수를 실행합니다.

In [7]:
# 도구 이름 → 실제 함수 매핑
TOOL_MAP: Dict[str, Any] = {t.name: t for t in TOOLS}


SYSTEM_PROMPT = """당신은 반도체 FAB의 MES 시스템 AI 어시스턴트입니다.
사용자의 질문을 분석하여 적절한 도구를 선택하고 호출하세요.

도구 선택 원칙:
- LOT ID(영문+숫자 혼합, 예: AB123456)가 포함된 경우 → LOT 관련 도구
- 장비 ID(예: M15A001, M16C002)가 포함된 경우 → get_equipment_status
- FAB 번호(M10~M16)만 있고 장비 ID가 없는 경우 → get_fab_equipment_summary
- 특정 공정명과 LOT 접두사가 있는 경우 → get_operation_history
- 위 모두 해당 없으면 → answer_general_question
"""


def run_tool_calling(query: str, verbose: bool = True) -> GraphState:
    """Tool Calling 방식으로 질문을 처리하는 메인 함수"""
    state = GraphState(query=query)

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=query)
    ]

    if verbose:
        print(f"[질문] {query}")
        print("-" * 60)

    # ① LLM 호출 → 어떤 도구를 호출할지 결정
    ai_response = llm_with_tools.invoke(messages)

    if not ai_response.tool_calls:
        if verbose:
            print("[결과] 도구 없이 직접 답변:")
            print(ai_response.content)
        state.answer = ai_response.content
        state.intent = "direct_answer"
        return state

    # ② tool_calls 파싱 & 실행
    for tc in ai_response.tool_calls:
        tool_name = tc["name"]
        tool_args = tc["args"]

        if verbose:
            print(f"[LLM 선택 도구] {tool_name}")
            print(f"[인자] {tool_args}")

        state.tool_calls_log.append(f"{tool_name}({tool_args})")
        state.intent = tool_name

        if tool_name not in TOOL_MAP:
            state.answer = f"알 수 없는 도구: {tool_name}"
            continue

        # ③ 도구 실행
        tool_result = TOOL_MAP[tool_name].invoke(tool_args)

        if verbose:
            print(f"[실행 결과 미리보기]")
            print(tool_result[:200] + "..." if len(tool_result) > 200 else tool_result)

        state.answer = tool_result
        state.skip_rag = (tool_name != "answer_general_question")

    return state


print("Tool Calling 실행기 준비 완료")

Tool Calling 실행기 준비 완료


## 7. LLM이 선택한 도구 스키마 확인

LLM에게 전달되는 도구 JSON 스키마를 확인합니다.

In [8]:
# LLM에 전달되는 도구 스키마 확인
for t in TOOLS:
    schema = t.args_schema.schema() if t.args_schema else {}
    props = list(schema.get('properties', {}).keys())
    print(f"도구명: {t.name}")
    print(f"  설명: {t.description.strip().splitlines()[0]}")
    print(f"  파라미터: {props}")
    print()

도구명: get_lot_slot_info
  설명: LOT의 현재 슬롯별 위치와 공정 정보를 조회합니다.
  파라미터: ['lot_id']

도구명: get_lot_setmonitor
  설명: LOT의 Set Monitoring(셋모) 계측 지시 정보를 조회합니다.
  파라미터: ['lot_id']

도구명: get_lot_future_hold
  설명: LOT의 Future Hold(퓨처홀드) 정보를 조회합니다.
  파라미터: ['lot_id']

도구명: get_lot_history
  설명: LOT의 공정 이력(이동 이력)을 시간 순으로 조회합니다.
  파라미터: ['lot_id']

도구명: get_equipment_status
  설명: 특정 장비 ID의 현재 상태, 챔버 정보, PORT 정보를 조회합니다.
  파라미터: ['eqp_id', 'fab']

도구명: get_fab_equipment_summary
  설명: FAB(공장) 전체 또는 특정 모델/구역의 장비 현황(대수, UP/DOWN)을 집계합니다.
  파라미터: ['fab', 'model', 'area']

도구명: get_operation_history
  설명: 특정 공정(operation)을 거친 LOT들의 이력을 조회합니다.
  파라미터: ['lot_prefix', 'operation_name', 'is_previous']

도구명: answer_general_question
  설명: 위의 어떤 도구도 해당하지 않는 일반적인 질문에 사용합니다.
  파라미터: ['query']



/var/folders/3_/_4rs10ps6577z2vsry31x7g00000gn/T/ipykernel_43188/2074481924.py:3: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  schema = t.args_schema.schema() if t.args_schema else {}


## 8. 시나리오 테스트 (LLM이 도구 선택)

> **주의**: 아래 셀들은 실제 LLM API 호출이 발생합니다. API 키가 필요합니다.

### 시나리오 1: LOT Slot 조회

In [9]:
state = run_tool_calling("AB123456 슬롯 정보 알려줘")
print()
display(Markdown(state.answer))

[질문] AB123456 슬롯 정보 알려줘
------------------------------------------------------------
[LLM 선택 도구] get_lot_slot_info
[인자] {'lot_id': 'AB123456'}
[실행 결과 미리보기]
##AB123456 - Slot 정보

| Slot | WF_ID | 공정 (FAB) | 시간 |
| --- | --- | --- | --- |
| 1 | WF001 | CVD 증착 (M15) | 2026-05-07 08:00:00 |
| 2 | WF002 | CVD 증착 (M15) | 2026-05-07 08:01:00 |
| 3 | WF003 | CMP...



##AB123456 - Slot 정보

| Slot | WF_ID | 공정 (FAB) | 시간 |
| --- | --- | --- | --- |
| 1 | WF001 | CVD 증착 (M15) | 2026-05-07 08:00:00 |
| 2 | WF002 | CVD 증착 (M15) | 2026-05-07 08:01:00 |
| 3 | WF003 | CMP 연마 (M15) | 2026-05-07 08:02:00 |
| 4 | *Empty* | - | - |
| 5 | WF005 | 검사 (M15) | 2026-05-07 08:03:00 |
| 6 | *Empty* | - | - |
| 7 | *Empty* | - | - |
| 8 | *Empty* | - | - |
| 9 | *Empty* | - | - |
| 10 | *Empty* | - | - |
| 11 | *Empty* | - | - |
| 12 | *Empty* | - | - |
| 13 | *Empty* | - | - |
| 14 | *Empty* | - | - |
| 15 | *Empty* | - | - |
| 16 | *Empty* | - | - |
| 17 | *Empty* | - | - |
| 18 | *Empty* | - | - |
| 19 | *Empty* | - | - |
| 20 | *Empty* | - | - |
| 21 | *Empty* | - | - |
| 22 | *Empty* | - | - |
| 23 | *Empty* | - | - |
| 24 | *Empty* | - | - |
| 25 | *Empty* | - | - |

### 시나리오 2: LOT SetMonitor

In [10]:
state = run_tool_calling("AB123456 계측 지시 확인해줘")
print()
display(Markdown(state.answer))

[질문] AB123456 계측 지시 확인해줘
------------------------------------------------------------
[LLM 선택 도구] get_lot_setmonitor
[인자] {'lot_id': 'AB123456'}
[실행 결과 미리보기]
##AB123456 - SetMonitor

| 공정 | Comment | 측정 Slot | 엔지니어 |
| --- | --- | --- | --- |
| CVD 증착 | 두께 측정 필요 | Slot 1,3,5 | 김엔지니어 |
| CMP 연마 | 평탄도 확인 | Slot 2,4 | 이엔지니어 |



##AB123456 - SetMonitor

| 공정 | Comment | 측정 Slot | 엔지니어 |
| --- | --- | --- | --- |
| CVD 증착 | 두께 측정 필요 | Slot 1,3,5 | 김엔지니어 |
| CMP 연마 | 평탄도 확인 | Slot 2,4 | 이엔지니어 |

### 시나리오 3: LOT Future Hold

In [11]:
state = run_tool_calling("AB123456 퓨처홀드 걸려있어?")
print()
display(Markdown(state.answer))

[질문] AB123456 퓨처홀드 걸려있어?
------------------------------------------------------------
[LLM 선택 도구] get_lot_future_hold
[인자] {'lot_id': 'AB123456'}
[실행 결과 미리보기]
##AB123456 - Future Hold

| 공정 | Comment | 엔지니어 |
| --- | --- | --- |
| 식각 공정 | 이상 감지 시 홀드 | 박엔지니어 |



##AB123456 - Future Hold

| 공정 | Comment | 엔지니어 |
| --- | --- | --- |
| 식각 공정 | 이상 감지 시 홀드 | 박엔지니어 |

### 시나리오 4: 장비 상태 조회

In [12]:
state = run_tool_calling("M15A001 지금 돌아가?")
print()
display(Markdown(state.answer))

[질문] M15A001 지금 돌아가?
------------------------------------------------------------
[LLM 선택 도구] get_equipment_status
[인자] {'eqp_id': 'M15A001', 'fab': 'M15'}
[실행 결과 미리보기]
## 장비 상태

| 항목 | 값 |
| --- | --- |
| EQP_ID | M15A001 |
| EQ_GROUP | CVD |
| MES_STAT_TYP | Up |
| EQP_STAT_CD | RUN |
| CHAMBER 수 | 2 |
| DOWN 수 | 1 |

| 챔버 ID | 상태 | EQP_STAT | LAST_EVENT |
| --- | ...



## 장비 상태

| 항목 | 값 |
| --- | --- |
| EQP_ID | M15A001 |
| EQ_GROUP | CVD |
| MES_STAT_TYP | Up |
| EQP_STAT_CD | RUN |
| CHAMBER 수 | 2 |
| DOWN 수 | 1 |

| 챔버 ID | 상태 | EQP_STAT | LAST_EVENT |
| --- | --- | --- | --- |
| M15A001_CH1 | Up | RUN | 2026-05-07 08:00:00 |
| M15A001_CH2 | Down | FAULT | 2026-05-06 20:00:00 |

### PORT 정보
| PORT | Transfer | 상태 |
| --- | --- | --- |
| P1 | 진행중 | Run |
| P2 | 대기 | Id |

### 시나리오 5: FAB 장비 집계

In [13]:
state = run_tool_calling("M15에서 CVD 장비 몇 대 다운이야?")
print()
display(Markdown(state.answer))

[질문] M15에서 CVD 장비 몇 대 다운이야?
------------------------------------------------------------
[LLM 선택 도구] get_fab_equipment_summary
[인자] {'fab': 'M15', 'model': 'CVD'}
[실행 결과 미리보기]
M15 조건에 맞는 장비 없음



M15 조건에 맞는 장비 없음

### 시나리오 6: 공정 이력

In [14]:
state = run_tool_calling("AB1 CVD 증착 이전 공정 뭐야?")
print()
if state.answer:
    display(Markdown(state.answer))

[질문] AB1 CVD 증착 이전 공정 뭐야?
------------------------------------------------------------
[LLM 선택 도구] get_operation_history
[인자] {'lot_prefix': 'AB1', 'operation_name': 'CVD', 'is_previous': True}
[실행 결과 미리보기]
'CVD' 공정 데이터 없음



'CVD' 공정 데이터 없음

### 시나리오 7: 일반 질문 (RAG fallback)

In [15]:
state = run_tool_calling("CVD 공정에서 두께 편차가 생기는 원인이 뭐야?")
print()
print(f"intent: {state.intent} | skip_rag: {state.skip_rag}")
if state.answer:
    display(Markdown(state.answer))

[질문] CVD 공정에서 두께 편차가 생기는 원인이 뭐야?
------------------------------------------------------------
[LLM 선택 도구] answer_general_question
[인자] {'query': 'CVD 공정에서 두께 편차 원인은?'}
[실행 결과 미리보기]
[RAG 검색 필요] 'CVD 공정에서 두께 편차 원인은?' → 지식 베이스 검색 후 LLM 답변 생성 단계로 진행

intent: answer_general_question | skip_rag: False


[RAG 검색 필요] 'CVD 공정에서 두께 편차 원인은?' → 지식 베이스 검색 후 LLM 답변 생성 단계로 진행

### 시나리오 8: 애매한 표현 (키워드 방식으로 어려운 케이스)

In [16]:
# 키워드 방식에서는 매칭 실패 가능성이 높은 케이스
ambiguous_queries = [
    "AB123456 혹시 계측 잡혀있어?",           # setmo (계측 키워드로 인식 가능)
    "M15 장비 중 멈춘 거 있어?",               # DOWN 현황
    "M15A002 왜 세워놨어?",                   # 장비 상태 (Down 이유)
    "AB123456 지금 어느 공정이야?",            # slot (공정 위치)
]

for q in ambiguous_queries:
    print(f"\n{'='*60}")
    state = run_tool_calling(q, verbose=True)
    print(f"→ intent: {state.intent}")


[질문] AB123456 혹시 계측 잡혀있어?
------------------------------------------------------------
[LLM 선택 도구] get_lot_setmonitor
[인자] {'lot_id': 'AB123456'}
[실행 결과 미리보기]
##AB123456 - SetMonitor

| 공정 | Comment | 측정 Slot | 엔지니어 |
| --- | --- | --- | --- |
| CVD 증착 | 두께 측정 필요 | Slot 1,3,5 | 김엔지니어 |
| CMP 연마 | 평탄도 확인 | Slot 2,4 | 이엔지니어 |
→ intent: get_lot_setmonitor

[질문] M15 장비 중 멈춘 거 있어?
------------------------------------------------------------
[LLM 선택 도구] get_fab_equipment_summary
[인자] {'fab': 'M15'}
[실행 결과 미리보기]
## M15 장비 현황 ( )

- 메인 장비 총: 3대 / 챔버: 2개

| MODEL | TOTAL | UP | DOWN |
| --- | --- | --- | --- |
| CMP_B | 1 | 1 | 0 |
| LPCVD_A | 2 | 1 | 1 |
→ intent: get_fab_equipment_summary

[질문] M15A002 왜 세워놨어?
------------------------------------------------------------
[LLM 선택 도구] get_equipment_status
[인자] {'eqp_id': 'M15A002', 'fab': 'M15'}
[실행 결과 미리보기]
## 장비 상태

| 항목 | 값 |
| --- | --- |
| EQP_ID | M15A002 |
| EQ_GROUP | CVD |
| MES_STAT_TYP | Down |
| EQP_STAT_CD | PM |
| CHAMBER 수 | 0 |


## 9. Mock LLM — API 키 없이 도구 선택 로직 확인

API 호출 없이 도구 선택 & 실행 흐름을 확인하는 시뮬레이터입니다.

In [17]:
# LLM 응답을 흉내내는 Mock — 실제 LLM이 어떤 JSON을 반환하는지 구조 확인용

MOCK_TOOL_DECISIONS = [
    {"query": "AB123456 슬롯 정보",          "tool": "get_lot_slot_info",        "args": {"lot_id": "AB123456"}},
    {"query": "AB123456 셋모 확인",           "tool": "get_lot_setmonitor",       "args": {"lot_id": "AB123456"}},
    {"query": "AB123456 퓨처홀드 있어?",      "tool": "get_lot_future_hold",      "args": {"lot_id": "AB123456"}},
    {"query": "AB123456 이력 조회",           "tool": "get_lot_history",          "args": {"lot_id": "AB123456"}},
    {"query": "M15A001 장비 상태",            "tool": "get_equipment_status",     "args": {"eqp_id": "M15A001", "fab": "M15"}},
    {"query": "M15 장비 현황",               "tool": "get_fab_equipment_summary", "args": {"fab": "M15"}},
    {"query": "M15 LPCVD_A 현황",            "tool": "get_fab_equipment_summary", "args": {"fab": "M15", "model": "LPCVD_A"}},
    {"query": "AB1 CVD 증착 이전 공정",       "tool": "get_operation_history",    "args": {"lot_prefix": "AB1", "operation_name": "CVD 증착", "is_previous": True}},
    {"query": "CVD 두께 편차 원인",           "tool": "answer_general_question",  "args": {"query": "CVD 두께 편차 원인"}},
]

print("Mock LLM Tool Calling 시뮬레이션")
print("=" * 60)

for case in MOCK_TOOL_DECISIONS:
    query  = case["query"]
    tool_name = case["tool"]
    args   = case["args"]

    print(f"\n질문: {query}")
    print(f"LLM 선택: {tool_name}({args})")

    result = TOOL_MAP[tool_name].invoke(args)
    preview = result.replace('\n', ' ')[:80]
    print(f"결과 미리보기: {preview}...")

Mock LLM Tool Calling 시뮬레이션

질문: AB123456 슬롯 정보
LLM 선택: get_lot_slot_info({'lot_id': 'AB123456'})
결과 미리보기: ##AB123456 - Slot 정보  | Slot | WF_ID | 공정 (FAB) | 시간 | | --- | --- | --- | --- |...

질문: AB123456 셋모 확인
LLM 선택: get_lot_setmonitor({'lot_id': 'AB123456'})
결과 미리보기: ##AB123456 - SetMonitor  | 공정 | Comment | 측정 Slot | 엔지니어 | | --- | --- | --- | -...

질문: AB123456 퓨처홀드 있어?
LLM 선택: get_lot_future_hold({'lot_id': 'AB123456'})
결과 미리보기: ##AB123456 - Future Hold  | 공정 | Comment | 엔지니어 | | --- | --- | --- | | 식각 공정 | ...

질문: AB123456 이력 조회
LLM 선택: get_lot_history({'lot_id': 'AB123456'})
결과 미리보기: ##AB123456 - 이력정보  | 순번 | 시간 | 이벤트 | 공정 ID | 공정명 | 수량 | 제품 | | --- | --- | --- |...

질문: M15A001 장비 상태
LLM 선택: get_equipment_status({'eqp_id': 'M15A001', 'fab': 'M15'})
결과 미리보기: ## 장비 상태  | 항목 | 값 | | --- | --- | | EQP_ID | M15A001 | | EQ_GROUP | CVD | | MES...

질문: M15 장비 현황
LLM 선택: get_fab_equipment_summary({'fab': 'M15'})
결과 미리보기: ## M15 장비 현황 ( )  - 메인 장비 총: 3대 / 챔버: 2개  | MODEL | TOTAL | UP |

## 10. Mock 결과 렌더링

In [18]:
# Mock LLM 결과를 Markdown으로 렌더링
for case in MOCK_TOOL_DECISIONS:
    print(f"\n{'='*60}")
    print(f"질문: {case['query']}")
    print(f"선택 도구: {case['tool']}")
    result = TOOL_MAP[case['tool']].invoke(case['args'])
    display(Markdown(result))


질문: AB123456 슬롯 정보
선택 도구: get_lot_slot_info


##AB123456 - Slot 정보

| Slot | WF_ID | 공정 (FAB) | 시간 |
| --- | --- | --- | --- |
| 1 | WF001 | CVD 증착 (M15) | 2026-05-07 08:00:00 |
| 2 | WF002 | CVD 증착 (M15) | 2026-05-07 08:01:00 |
| 3 | WF003 | CMP 연마 (M15) | 2026-05-07 08:02:00 |
| 4 | *Empty* | - | - |
| 5 | WF005 | 검사 (M15) | 2026-05-07 08:03:00 |
| 6 | *Empty* | - | - |
| 7 | *Empty* | - | - |
| 8 | *Empty* | - | - |
| 9 | *Empty* | - | - |
| 10 | *Empty* | - | - |
| 11 | *Empty* | - | - |
| 12 | *Empty* | - | - |
| 13 | *Empty* | - | - |
| 14 | *Empty* | - | - |
| 15 | *Empty* | - | - |
| 16 | *Empty* | - | - |
| 17 | *Empty* | - | - |
| 18 | *Empty* | - | - |
| 19 | *Empty* | - | - |
| 20 | *Empty* | - | - |
| 21 | *Empty* | - | - |
| 22 | *Empty* | - | - |
| 23 | *Empty* | - | - |
| 24 | *Empty* | - | - |
| 25 | *Empty* | - | - |


질문: AB123456 셋모 확인
선택 도구: get_lot_setmonitor


##AB123456 - SetMonitor

| 공정 | Comment | 측정 Slot | 엔지니어 |
| --- | --- | --- | --- |
| CVD 증착 | 두께 측정 필요 | Slot 1,3,5 | 김엔지니어 |
| CMP 연마 | 평탄도 확인 | Slot 2,4 | 이엔지니어 |


질문: AB123456 퓨처홀드 있어?
선택 도구: get_lot_future_hold


...

## M15 장비 현황 (LPCVD_A )

- 메인 장비 총: 2대 / 챔버: 2개

| MODEL | TOTAL | UP | DOWN |
| --- | --- | --- | --- |
| LPCVD_A | 2 | 1 | 1 |


질문: AB1 CVD 증착 이전 공정
선택 도구: get_operation_history


## CVD 증착 이전 공정 공정 이력

| LOT_ID | WF_QTY | CTN_DESC | 상태 | 시간 | FLOW_ID |
| --- | --- | --- | --- | --- | --- |
| AB1002 | 24 | CVD 증착 | COMPLETE | 2026-05-06 20:00 | FLOW_A |
| AB1001 | 25 | 포토 리소그래피 | COMPLETE | 2026-05-07 06:00 | FLOW_A |


질문: CVD 두께 편차 원인
선택 도구: answer_general_question


[RAG 검색 필요] 'CVD 두께 편차 원인' → 지식 베이스 검색 후 LLM 답변 생성 단계로 진행

## 11. 키워드 방식 vs Tool Calling 비교

| 질문 | 키워드 방식 | Tool Calling |
|------|------------|-------------|
| `AB123456 슬롯 정보` | `'슬롯' in query` → OK | `get_lot_slot_info` 선택 |
| `AB123456 혹시 계측 잡혀있어?` | `'계측' in query` → OK | LLM이 의도 파악 → `get_lot_setmonitor` |
| `AB123456 지금 어느 공정이야?` | 키워드 없음 → 이력 fallback | `get_lot_slot_info` 정확 선택 |
| `M15 멈춘 장비 있어?` | FAB 감지, DOWN 키워드 없음 → 실패 가능 | `get_fab_equipment_summary` 선택 |
| `M15A002 왜 세워놨어?` | 상태 키워드 없음 → 매칭 어려움 | `get_equipment_status` 선택 |